In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# LOAD ALL DATA
defense   = pd.read_excel("Defense.xlsx")
kicking   = pd.read_excel("Kicking.xlsx")
kickoffs  = pd.read_excel("Kickoffs.xlsx")
passing   = pd.read_excel("Passing.xlsx")
punting   = pd.read_excel("Punting.xlsx")
receiving = pd.read_excel("Receiving.xlsx")
returns   = pd.read_excel("Returns.xlsx")
rushing   = pd.read_excel("Rushing.xlsx")
winloss   = pd.read_excel("WinLoss.xlsx")


# CLEAN NON-NUMERIC ENTRIES
def clean_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = (
                df[c]
                .astype(str)
                .str.replace(r'[^0-9\.-]', '', regex=True)
                .replace('', '0')
            )
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df

# CLEAN PLACEHOLDERS
# Passing
clean_numeric(passing,
    ["Comp","Att","Yds","TD","Int","1stD","Sacks","YdsL","Fumb","FumL"]
)

# Rushing
clean_numeric(rushing,
    ["Att","Yds","TD","1stD","Fumb","FumL","Long"]
)

# Receiving
clean_numeric(receiving,
    ["Rec","Tgt","Yds","TD","1stD","Fumb","FumL"]
)

# Kicking
clean_numeric(kicking,
    ["FGM19","FGA19","FGM29","FGA29","FGM39","FGA39",
     "FGM49","FGA49","FGM50","FGA50","FGM","FGA","XPM","XPA"]
)

# Kickoffs
clean_numeric(kickoffs, ["KO","Yds","FC","TB","KR","KR TD"])

# Punting
clean_numeric(punting, ["Punt","Yds","In20","FC","TB","BP","PR"])

# Returns
clean_numeric(returns, ["KR","Yds","TD","PR"])

# Defense
clean_numeric(defense,
    ["Solo","Ast","Total","Sack","YdsL","Stuff","Int","IntYds",
     "IntTD","FF","FRTD","PD","Sfty"]
)

# AGGREGATIONS
# ---------------- Passing ----------------
team_passing = passing.groupby(["Team", "Week"]).agg(
    Pass_Comp=("Comp", "sum"),
    Pass_Att=("Att", "sum"),
    Pass_Yds=("Yds", "sum"),
    Pass_TD=("TD", "sum"),
    Pass_Int=("Int", "sum"),
    Pass_1stD=("1stD", "sum"),
    Pass_Sacks=("Sacks", "sum"),
    Pass_SackYds=("YdsL", "sum"),
    Pass_Fumb=("Fumb", "sum"),
    Pass_FumL=("FumL", "sum")
)
team_passing["Pass_Pct"] = team_passing["Pass_Comp"] / team_passing["Pass_Att"]
team_passing["Pass_YPA"] = team_passing["Pass_Yds"] / team_passing["Pass_Att"]


# ---------------- Rushing ----------------
team_rushing = rushing.groupby(["Team", "Week"]).agg(
    Rush_Att=("Att","sum"),
    Rush_Yds=("Yds","sum"),
    Rush_TD=("TD","sum"),
    Rush_1stD=("1stD","sum"),
    Rush_Fumb=("Fumb","sum"),
    Rush_FumL=("FumL","sum"),
    Rush_Long=("Long","max")
)
team_rushing["Rush_YPA"] = team_rushing["Rush_Yds"] / team_rushing["Rush_Att"]


# ---------------- Receiving ----------------
team_receiving = receiving.groupby(["Team", "Week"]).agg(
    Rec_Rec=("Rec","sum"),
    Rec_Tgt=("Tgt","sum"),
    Rec_Yds=("Yds","sum"),
    Rec_TD=("TD","sum"),
    Rec_1stD=("1stD","sum"),
    Rec_Fumb=("Fumb","sum"),
    Rec_FumL=("FumL","sum")
)
team_receiving["Rec_YPR"] = team_receiving["Rec_Yds"] / team_receiving["Rec_Rec"]


# ---------------- Kicking ----------------
team_kicking = kicking.groupby(["Team", "Week"]).agg(
    Kick_FGM19=("FGM19","sum"),
    Kick_FGA19=("FGA19","sum"),
    Kick_FGM29=("FGM29","sum"),
    Kick_FGA29=("FGA29","sum"),
    Kick_FGM39=("FGM39","sum"),
    Kick_FGA39=("FGA39","sum"),
    Kick_FGM49=("FGM49","sum"),
    Kick_FGA49=("FGA49","sum"),
    Kick_FGM50=("FGM50","sum"),
    Kick_FGA50=("FGA50","sum"),
    Kick_FGM=("FGM","sum"),
    Kick_FGA=("FGA","sum"),
    Kick_XPM=("XPM","sum"),
    Kick_XPA=("XPA","sum")
)


# ---------------- Kickoffs ----------------
team_kickoffs = kickoffs.groupby(["Team", "Week"]).agg(
    KO_KO=("KO","sum"),
    KO_Yds=("Yds","sum"),
    KO_FC=("FC","sum"),
    KO_TB=("TB","sum"),
    KO_KR=("KR","sum"),
    KO_KRTD=("KR TD","sum")
)
team_kickoffs["KO_YdsPerKR"] = (
    team_kickoffs["KO_Yds"] / team_kickoffs["KO_KR"]
).replace([np.inf, -np.inf], 0)


# ---------------- Punting ----------------
team_punting = punting.groupby(["Team", "Week"]).agg(
    Punt_Punt=("Punt","sum"),
    Punt_Yds=("Yds","sum"),
    Punt_In20=("In20","sum"),
    Punt_FC=("FC","sum"),
    Punt_TB=("TB","sum"),
    Punt_BP=("BP","sum"),
    Punt_PR=("PR","sum")
)
team_punting["Punt_YdsPerPR"] = (
    team_punting["Punt_Yds"] / team_punting["Punt_PR"]
).replace([np.inf, -np.inf], 0)


# ---------------- Returns ----------------
team_returns = returns.groupby(["Team", "Week"]).agg(
    Ret_KR=("KR","sum"),
    Ret_Yds=("Yds","sum"),
    Ret_TD=("TD","sum"),
    Ret_PR=("PR","sum")
)
team_returns["Ret_AvgPR"] = (
    team_returns["Ret_Yds"] / team_returns["Ret_PR"]
).replace([np.inf, -np.inf], 0)


# ---------------- Defense ----------------
team_defense = defense.groupby(["Team", "Week"]).agg(
    Def_Solo=("Solo","sum"),
    Def_Ast=("Ast","sum"),
    Def_Total=("Total","sum"),
    Def_Sack=("Sack","sum"),
    Def_SackYds=("YdsL","sum"),
    Def_Stuff=("Stuff","sum"),
    Def_Int=("Int","sum"),
    Def_IntYds=("IntYds","sum"),
    Def_IntTD=("IntTD","sum"),
    Def_FF=("FF","sum"),
    Def_FRTD=("FRTD","sum"),
    Def_PD=("PD","sum"),
    Def_Sfty=("Sfty","sum")
)

# MERGE ALL TEAM TABLES
team_stats = (
    team_passing
    .merge(team_rushing, on=["Team", "Week"], how="outer")
    .merge(team_receiving, on=["Team", "Week"], how="outer")
    .merge(team_kicking, on=["Team", "Week"], how="outer")
    .merge(team_kickoffs, on=["Team", "Week"], how="outer")
    .merge(team_punting, on=["Team", "Week"], how="outer")
    .merge(team_returns, on=["Team", "Week"], how="outer")
    .merge(team_defense, on=["Team", "Week"], how="outer")
    .merge(winloss, on=["Team", "Week"], how="outer")
    .reset_index()
)

team_stats = team_stats.sort_index()

print(team_stats.head())

   index Team  Week  Pass_Comp  Pass_Att  Pass_Yds  Pass_TD  Pass_Int  \
0      0  ARI     1       21.0      29.0     163.0      2.0       0.0   
1      1  ARI     2       17.0      25.0     220.0      1.0       1.0   
2      2  ARI     3       22.0      35.0     159.0      1.0       0.0   
3      3  ARI     4       27.0      41.0     200.0      2.0       2.0   
4      4  ARI     5       23.0      32.0     220.0      0.0       0.0   

   Pass_1stD  Pass_Sacks  ...  Def_Stuff  Def_Int  Def_IntYds  Def_IntTD  \
0       11.0         5.0  ...        0.0      0.0         0.0        0.0   
1       10.0         1.0  ...        0.0      0.0         0.0        0.0   
2        8.0         1.0  ...        0.0      1.0        18.0        0.0   
3       13.0         6.0  ...        0.0      0.0         0.0        0.0   
4        8.0         3.0  ...        0.0      1.0         0.0        0.0   

   Def_FF  Def_FRTD  Def_PD  Def_Sfty          Team Name  W/L  
0     0.0       0.0     5.0       0.0  A

In [29]:
readable_names = {
    # Passing
    "Pass_Comp": "pass_completions",
    "Pass_Att": "pass_attempts",
    "Pass_Yds": "passing_yards",
    "Pass_TD": "passing_touchdowns",
    "Pass_Int": "interceptions_thrown",
    "Pass_1stD": "passing_first_downs",
    "Pass_Sacks": "times_sacked",
    "Pass_SackYds": "sack_yards_lost",
    "Pass_Fumb": "pass_fumbles",
    "Pass_FumL": "pass_fumbles_lost",
    "Pass_Pct": "completion_percentage",
    "Pass_YPA": "yards_per_attempt",

    # Rushing
    "Rush_Att": "rush_attempts",
    "Rush_Yds": "rushing_yards",
    "Rush_TD": "rushing_touchdowns",
    "Rush_1stD": "rushing_first_downs",
    "Rush_Fumb": "rush_fumbles",
    "Rush_FumL": "rush_fumbles_lost",
    "Rush_Long": "longest_rush",
    "Rush_YPA": "yards_per_rush",

    # Receiving
    "Rec_Rec": "receptions",
    "Rec_Tgt": "targets",
    "Rec_Yds": "receiving_yards",
    "Rec_TD": "receiving_touchdowns",
    "Rec_1stD": "receiving_first_downs",
    "Rec_Fumb": "receiving_fumbles",
    "Rec_FumL": "receiving_fumbles_lost",
    "Rec_YPR": "yards_per_reception",

    # Kicking
    "Kick_FGM19": "fg_made_(0-19)",
    "Kick_FGA19": "fg_attempted_(0-19)",
    "Kick_FGM29": "fg_made_(20-29)",
    "Kick_FGA29": "fg_attempted_(20-29)",
    "Kick_FGM39": "fg_made_(30-39)",
    "Kick_FGA39": "fg_attempted_(30-39)",
    "Kick_FGM49": "fg_made_(40-49)",
    "Kick_FGA49": "fg_attempted_(40-49)",
    "Kick_FGM50": "fg_made_(50+)",
    "Kick_FGA50": "fg_attempted_(50+)",
    "Kick_FGM": "fg_made_(total)",
    "Kick_FGA": "fg_attempted_(total)",
    "Kick_XPM": "extra_points_made",
    "Kick_XPA": "extra_points_attempted",

    # Kickoffs
    "KO_KO": "kickoffs",
    "KO_Yds": "kickoff_yards",
    "KO_FC": "kickoff_fair_catches",
    "KO_TB": "touchbacks",
    "KO_KR": "kick_returns_allowed",
    "KO_KRTD": "kick_return_touchdowns_allowed",
    "KO_YdsPerKR": "kick_return_yards_allowed_per_return",

    # Punting
    "Punt_Punt": "punts",
    "Punt_Yds": "punt_yards",
    "Punt_In20": "punts_inside_20",
    "Punt_FC": "punt_fair_catches",
    "Punt_TB": "punt_touchbacks",
    "Punt_BP": "blocked_punts",
    "Punt_PR": "punt_returns_allowed",
    "Punt_YdsPerPR": "punt_return_yards_allowed_per_return",

    # Returns
    "Ret_KR": "kick_returns",
    "Ret_Yds": "kick_return_yards",
    "Ret_TD": "kick_return_touchdowns",
    "Ret_PR": "punt_returns",
    "Ret_AvgPR": "average_punt_return_yards",

    # Defense
    "Def_Solo": "solo_tackles",
    "Def_Ast": "assisted_tackles",
    "Def_Total": "total_tackles",
    "Def_Sack": "sacks",
    "Def_SackYds": "sack_yards",
    "Def_Stuff": "run_stuffs",
    "Def_Int": "interceptions",
    "Def_IntYds": "interception_return_yards",
    "Def_IntTD": "interception_touchdowns",
    "Def_FF": "forced_fumbles",
    "Def_FRTD": "fumble_return_touchdowns",
    "Def_PD": "passes_defended",
    "Def_Sfty": "safeties",

    # Win/Loss
    "Team Name": "team_name_full",
    "Team": "team_abbr",
    "W/L": "win_loss",
    "Week": "game_week"
}
team_stats = team_stats.rename(columns=readable_names)
print(team_stats.head())

   index team_abbr  game_week  pass_completions  pass_attempts  passing_yards  \
0      0       ARI          1              21.0           29.0          163.0   
1      1       ARI          2              17.0           25.0          220.0   
2      2       ARI          3              22.0           35.0          159.0   
3      3       ARI          4              27.0           41.0          200.0   
4      4       ARI          5              23.0           32.0          220.0   

   passing_touchdowns  interceptions_thrown  passing_first_downs  \
0                 2.0                   0.0                 11.0   
1                 1.0                   1.0                 10.0   
2                 1.0                   0.0                  8.0   
3                 2.0                   2.0                 13.0   
4                 0.0                   0.0                  8.0   

   times_sacked  ...  run_stuffs  interceptions  interception_return_yards  \
0           5.0  ...      

In [30]:
col = team_stats.pop("team_name_full")
team_stats.insert(1, "team_name_full", col)
pd.DataFrame.to_excel(team_stats, "Team_Stats_By_Week.xlsx", index=False)

In [31]:
# checking df for duplicates
team_stats.duplicated().sum()

0